Building a Data Pipeline with dlt

In [1]:
!pip -q install dlt[duckdb]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.9/348.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 4.3 MB/s eta 0:00:00


In [2]:
import dlt
import dlt
from itertools import islice
from dlt.sources.rest_api import rest_api_source

In [3]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })


In [5]:
import os
# Force dlt to ignore the Colab environment's broken secret provider
os.environ["DLT_CONFIG_PROVIDERS__0__TYPE"] = "toml"

In [8]:
os.environ["DLT_CONFIG_PROVIDERS__DISABLE_COLAB_SECRETS"] = "True"

# 2. Tell dlt to look for the secrets file in your current directory
os.environ["DLT_CONFIG_PROVIDERS__0__TYPE"] = "toml"
os.environ["DLT_CONFIG_PROVIDERS__0__LOCATION"] = "./.dlt/secrets.toml"

import dlt

pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optiona)
)

TomlProviderReadException: A problem encountered when loading secrets.toml from paths ['./.dlt/secrets.toml', '/var/dlt/secrets.toml']:
Requesting secret secrets.toml timed out. Secrets can only be fetched when running from the Colab UI.